## Data From Vendor

In [1]:
import pandas as pd 

def parse(filename):
    cols = ["date","country","event","currency","previous","estimate","actual","impact","source"]
    df = pd.read_csv(filename, names=cols, index_col=["date","event"])["actual"].groupby("event").last().rename("Bond Yield")
    df.index = df.index.str.extract(r'(\d+)', expand=False).astype(int).rename("Maturity")
    return df.sort_index()

country = "KR"
year = 2025
# wget https://data-api.londonstrategicedge.com/download/economic-calendar/KR/2025.csv.gz -O- | zcat | grep "KTB Auction">  x01_gvt_KR.csv

## Official tbill auction data

### South Korea

In South Korea, every tbill issue is assigned a code. 

You can ask gemini to translate and extract the data from their website.

* Tbill issuance in 2025-12 : https://ktb.mofe.go.kr/stsBidResult.do
* Tbill auction result : https://ktb.moef.go.kr/bidResult.do
  (Old data : https://ktb.mofe.go.kr/stsBidResult.do)

* 2025-12 auction result of most recent issuance of Korean Tbill

| Maturity(Approximate)    |  Code (format : Coupon rate - Expiry YYMM ) | Weighted Average Interest Rate | Best Bid |
|--------------------------|---------------------------------------------|--------------------------------|----------|
|  2-year                  | KTB 02250-2709(25-6)                        | 2.876%                         |  2.830%  |
|  3-year                  | KTB 02750-2812(25-10)                       | 2.970%                         |  2.960%  |
|  5-year                  | KTB 02500-3009(25-8)                        | 3.245%                         |  3.215%  |
| 10-year                  | KTB 03250-3512(25-11)                       | 3.410%                         |  3.375%  |
| 20-year                  | KTB 02750-4509(25-9)                        | 3.365%                         |  3.330%  |
| 30-year                  | KTB 02625-5509(25-7)                        | 3.225%                         |  3.225%  |
| 50-year                  | KTB 02750-7409(24-11)                       | 3.140%                         |  2.000%  |

Banks trade in this market. (Agents with low funding cost (LIBOR))

## Consistency check

In [2]:
df_tbill = pd.DataFrame({
    "Korean Ministry of Finance and Economy": pd.Series({ 2: 2.876, 3: 2.970, 5: 3.245,10: 3.410,20: 3.365,30: 3.225,50: 3.140,}),
    "External data source":parse(f"x01_gvt_{country}.csv")})
assert((df_tbill.iloc[:,0] == df_tbill.iloc[:,1]).all())
df_tbill

,Korean Ministry of Finance and Economy,External data source
2,2.876,2.876
3,2.970,2.970
5,3.245,3.245
10,3.410,3.410
20,3.365,3.365
30,3.225,3.225
50,3.140,3.140


## Interpretation

### How KTB auction works 

* They publish the bond spec in auction notice : https://ktb.moef.go.kr/isuNdReprchsPblanc.do

Payment Schedule
* Principal: Lump-sum
* Coupon: Biannual

Auction mechanism 
* https://papers.ssrn.com/sol3/papers.cfm?abstract_id=2788365
* Multi-unit auction
* Every winning bidder receives yield they bid, approximately.
* An agent submits max. 7 bids in a single auction.
* lot size is 1 billion KRW.

In [3]:

def bond_pricing(notional, payment_per_year, coupon_rate, accepted_yield, issue_date, maturity_date, settlement_date):
    months = {1:12, 2:6, 4:3, 12:1}[payment_per_year]
    terms = pd.date_range(issue_date, maturity_date, freq=pd.DateOffset(months=months))
    df_cashflow = pd.DataFrame({
    "date": terms,
    "index": range(len(terms)),
    "cashflow": float(0.0),
    }).set_index("date")
    df_cashflow["discount"] = df_cashflow["index"].apply(lambda n : 1/(1+accepted_yield/payment_per_year)**n)
    df_cashflow.loc[maturity_date,"cashflow"] += notional
    df_cashflow.loc[terms[1:],"cashflow"] += notional * coupon_rate / payment_per_year
    issue_date_pv = df_cashflow.eval("cashflow * discount").sum()
    df_cashflow.loc[issue_date, "cashflow"] -= issue_date_pv

    coupon_date = terms[1]
    assert settlement_date < coupon_date
    if settlement_date == issue_date:
        dirty_price = issue_date_pv
    elif settlement_date > issue_date:
        a = (coupon_date - settlement_date).days
        b = (coupon_date - issue_date).days
        PV_at_coupon_date = issue_date_pv * (1+accepted_yield/payment_per_year)
        dirty_price = PV_at_coupon_date / (1+accepted_yield/payment_per_year * a/b)
    elif settlement_date < issue_date:
        a = (issue_date - settlement_date).days
        b = (issue_date - (issue_date - pd.DateOffset(months=months))).days
        dirty_price = issue_date_pv / (1+accepted_yield/payment_per_year * a/b) # Is it a standard way to do this? 
    else:
        raise Exception()

    return df_cashflow, dirty_price


In [5]:
# 2-year Korean T-bill
# https://www.bok.or.kr/portal/bbs/P0001794/view.do?nttId=10094827&menuNo=200364

notional = 1e9 # KRW 1B
coupon_rate = 2.25 * 0.01
payment_per_year = 2 # biannual
accepted_yield = 2.876 * 0.01 # weighted average, NOT continuously compounded!

trade_date = pd.to_datetime("2025-12-01")
settlement_date = pd.to_datetime("2025-12-02")
issue_date = pd.to_datetime("2025-09-10")
maturity_date = pd.to_datetime("2027-09-10")

df_cashflow, dirty_price = bond_pricing(notional, payment_per_year, coupon_rate, accepted_yield, issue_date, maturity_date, settlement_date)

print(f"Face value : {round(notional)}")
print(f"Dirty price : {int(dirty_price)}")
print("")
print("Cashflow:")
df_cashflow

Face value : 1000000000
Dirty price : 994381608

Cashflow:


,index,cashflow,discount
date,,,
2025-09-10,0,-9.879175e+08,1.000000
2026-03-10,1,1.125000e+07,0.985824
2026-09-10,2,1.125000e+07,0.971849
2027-03-10,3,1.125000e+07,0.958072
2027-09-10,4,1.011250e+09,0.944490


## Summary

Average bankers agreed to send KRW 0.994B to Korean Govt until 2025-12-02 market close, for a long position of 02250-2709 contract of KRW 1B face value.

So, if
  * A customer had deposited KRW 0.994B in the bank before 2025-12-01 
  * And the bank had promised to pay 0.01B every 6 month,
  * And the bank had promised to return KRW 1B by 2027-09-10 to the customer,
  * Then purchaing this bond can be useful for the bank.

So many numbers. 
For hedging, we need a single number that we want to make zero. 
This is the reason why we calculate the duration of bonds.